# Write a solution to find the second highest distinct salary from the Employee table. If there is no second highest salary, return null (return None in Pandas).

The result format is in the following example.

 

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType

spark = SparkSession.builder.getOrCreate()

# Schema
employee_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("salary", IntegerType(), True),
])

# Example 1 data
employee_data_1 = [
    (1, 100),
    (2, 200),
    (3, 300),
]

employee_df_1 = spark.createDataFrame(employee_data_1, schema=employee_schema)

# Example 2 data
employee_data_2 = [
    (1, 100),
]

employee_df_2 = spark.createDataFrame(employee_data_2, schema=employee_schema)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/06 08:36:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
employee_df_1.show()
employee_df_2.show()

+---+------+
| id|salary|
+---+------+
|  1|   100|
|  2|   200|
|  3|   300|
+---+------+

+---+------+
| id|salary|
+---+------+
|  1|   100|
+---+------+



In [5]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [6]:
# performance not good at scale use distinct and order by
window_spec=Window.orderBy(col("salary").desc())
employee_df_1.withColumn("rank",dense_rank().over(window_spec)).where(col("rank")==2).show()

26/01/06 08:36:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 08:36:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 08:36:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 08:36:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/06 08:36:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+------+----+
| id|salary|rank|
+---+------+----+
|  2|   200|   2|
+---+------+----+



In [10]:
#fails for 2nd df because spark spark desnot guarantee order after limit(2)
employee_df_1.select("salary").distinct().orderBy("salary",ascending=False).limit(2).orderBy("salary").limit(1).show()

+------+
|salary|
+------+
|   200|
+------+



In [ ]:
# works on tb scale 
# get highest salary then find all data less than highest then max of that
highest_salary=employee_df_1.agg(max("salary").alias("max_salary")).collect()[0]["max_salary"]


300

In [28]:
employee_df_1.filter(col("salary")<employee_df_1.agg(max("salary")).collect()[0][0]).agg(max("salary").alias("sec_highest")).show()

+-----------+
|sec_highest|
+-----------+
|        200|
+-----------+

